In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
import time
import warnings

warnings.filterwarnings('ignore')

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
    torch.cuda.set_device(0)
    device = 'cuda'
else:
    device = 'cpu'

print(f"\ndevice: {device.upper()}")


df = pd.read_csv('Breast Cancer METABRIC.csv')

target_col = 'Tumor Stage'
original_count = len(df)
df = df[df[target_col] != 4].copy()
removed_count = original_count - len(df)
df = df.dropna(subset=[target_col])
leakage_features = [
    'Overall Survival (Months)',
    'Overall Survival Status',
    'Patient\'s Vital Status',
    'Relapse Free Status (Months)',
    'Relapse Free Status',
    'Nottingham prognostic index',
]
removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)
missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"{len(high_missing)} features with high missing rate have been removed")

exclude_kw = ['stage', 'unnamed', 'patient id', 'patient\'s vital',
              'survival', 'relapse', 'status']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)


X = df[feature_cols].copy()
y = df[target_col].copy()

# Handle missing values
for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)


# Label encoding

for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

for i, label in enumerate(le_target.classes_):
    print(f"  Stage {label} → {i}")

print(f"Feature matrix: {X.shape}")

# Feature selection
K_FEATURES = 15
selector = SelectKBest(
    score_func=lambda X, y: mutual_info_classif(X, y, random_state=42),
    k=min(K_FEATURES, len(feature_cols))
)
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(feature_scores.head(20).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (score: {score:.4f})")

X_selected = X[selected_features].copy()


# Data splitting
class_counts = pd.Series(y_encoded).value_counts()
print("\nNumber of samples per stage:")
for stage_idx, count in class_counts.items():
    stage_name = le_target.classes_[stage_idx]
    print(f"  Stage {stage_name} ({stage_idx}): {count}")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y_encoded

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("\nTraining set distribution (before SMOTE):")
for s, c in pd.Series(y_train).value_counts().sort_index().items():
    print(f"  Stage {le_target.classes_[s]} ({s}): {c}")


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE
min_class_samples = pd.Series(y_train).value_counts().min()
k_neighbors = min(5, min_class_samples - 1)

if k_neighbors >= 1:
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

    for s, c in pd.Series(y_train_bal).value_counts().sort_index().items():
        print(f"  Stage {le_target.classes_[s]} ({s}): {c}")
else:
    X_train_bal, y_train_bal = X_train_scaled, y_train

# TabNet + XGBoost + Logistic Regression
tabnet_model = TabNetClassifier(
    n_d=32,
    n_a=32,
    n_steps=3,
    gamma=1.3,
    lambda_sparse=1e-3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.02),
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax',
    seed=42,
    device_name=device,
    verbose=10
)


print("Training TabNet...")


start_time = time.time()

tabnet_model.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test_scaled, y_test)],
    eval_name=['validation'],
    eval_metric=['accuracy'],
    max_epochs=100,
    patience=20,
    batch_size=256,
    virtual_batch_size=128,
    drop_last=False
)

training_time = time.time() - start_time

print(f" TabNet training completed")
print(f"Total training time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Actual training epochs: {len(tabnet_model.history['loss'])}")


eval_set = [(X_train_bal, y_train_bal), (X_test_scaled, y_test)]
eval_names = ['train', 'validation']

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.01,
    max_depth=5,
    subsample=1.0,
    colsample_bytree=1.0,
    objective='multi:softprob',
    num_class=len(np.unique(y_train_bal)),
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)


print("Training XGBoost...")


start_time_xgb = time.time()

xgb_model.fit(
    X_train_bal, y_train_bal,
    eval_set=eval_set,
    verbose=10
)

xgb_training_time = time.time() - start_time_xgb

evals_result = xgb_model.evals_result()
train_logloss = evals_result['validation_0']['mlogloss']
val_logloss = evals_result['validation_1']['mlogloss']

best_iteration = np.argmin(val_logloss) + 1



tabnet_train_proba = tabnet_model.predict_proba(X_train_bal)
xgb_train_proba = xgb_model.predict_proba(X_train_bal)

print(f"  TabNet output shape: {tabnet_train_proba.shape}")
print(f"  XGBoost output shape: {xgb_train_proba.shape}")


meta_train_features = np.hstack([tabnet_train_proba, xgb_train_proba])
print(f"  Meta-learner input shape: {meta_train_features.shape}")

meta_learner = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs',
    verbose=1
)


start_time_meta = time.time()

meta_learner.fit(meta_train_features, y_train_bal)

meta_training_time = time.time() - start_time_meta

print(f"Meta-learner training completed")
print(f"  Training time: {meta_training_time:.2f} seconds")
print(f"  Number of iterations: {meta_learner.n_iter_[0]}")


#  Cross-validation


from sklearn.base import BaseEstimator, ClassifierMixin

class StackedEnsemble(BaseEstimator, ClassifierMixin):

    def __init__(self, device='cpu', random_state=42):
        self.device = device
        self.random_state = random_state

    def fit(self, X, y):

        self.tabnet = TabNetClassifier(
            n_d=32, n_a=32, n_steps=3, gamma=1.3, lambda_sparse=1e-3,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=0.02),
            seed=self.random_state,
            device_name=self.device,
            verbose=0
        )
        self.tabnet.fit(
            X.values if hasattr(X, 'values') else X, y,
            max_epochs=100, patience=20, batch_size=256,
            virtual_batch_size=128
        )


        self.xgb = xgb.XGBClassifier(
            n_estimators=200, learning_rate=0.01, max_depth=5,
            subsample=1.0, objective='multi:softprob',
            num_class=len(np.unique(y)),
            random_state=self.random_state, verbosity=0
        )
        self.xgb.fit(X, y)


        X_arr = X.values if hasattr(X, 'values') else X
        tabnet_proba = self.tabnet.predict_proba(X_arr)
        xgb_proba = self.xgb.predict_proba(X)
        meta_features = np.hstack([tabnet_proba, xgb_proba])


        self.meta = LogisticRegression(
            C=1.0, max_iter=1000, random_state=self.random_state,
            multi_class='multinomial', solver='lbfgs', verbose=0
        )
        self.meta.fit(meta_features, y)

        return self

    def predict(self, X):
        X_arr = X.values if hasattr(X, 'values') else X
        tabnet_proba = self.tabnet.predict_proba(X_arr)
        xgb_proba = self.xgb.predict_proba(X)
        meta_features = np.hstack([tabnet_proba, xgb_proba])
        return self.meta.predict(meta_features)


ensemble_cv = StackedEnsemble(device=device, random_state=42)

cv_start_time = time.time()
cv_scores = cross_val_score(
    ensemble_cv, X_selected, y_encoded,
    cv=5, scoring='accuracy', n_jobs=1
)
cv_time = time.time() - cv_start_time


for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i:2d}: {score:.4f}")
print(f"\n{'='*40}")
print(f"  Average accuracy: {cv_scores.mean():.4f}")
print(f"  Standard deviation: {cv_scores.std():.4f}")
print(f"  Confidence interval: [{cv_scores.mean()-1.96*cv_scores.std():.4f}, {cv_scores.mean()+1.96*cv_scores.std():.4f}]")


#  Feature importance analysis


xgb_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nXGBoost:")
print(xgb_importance.to_string(index=False))


tabnet_feature_importance = tabnet_model.feature_importances_
tabnet_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': tabnet_feature_importance
}).sort_values('importance', ascending=False)

print("TabNet:")
print(tabnet_importance.to_string(index=False))

# Model evaluation


tabnet_train_test_proba = tabnet_model.predict_proba(X_train_scaled)
xgb_train_test_proba = xgb_model.predict_proba(X_train_scaled)
meta_train_test_features = np.hstack([tabnet_train_test_proba, xgb_train_test_proba])
y_train_pred = meta_learner.predict(meta_train_test_features)

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)


tabnet_test_proba = tabnet_model.predict_proba(X_test_scaled)
xgb_test_proba = xgb_model.predict_proba(X_test_scaled)
meta_test_features = np.hstack([tabnet_test_proba, xgb_test_proba])
y_test_pred = meta_learner.predict(meta_test_features)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)


print("\n" + "=" * 70)
print(" " * 20 + "Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy:>15.4f} {test_accuracy:>15.4f} {abs(train_accuracy-test_accuracy):>15.4f}")
print(f"{'Precision':<15} {train_precision:>15.4f} {test_precision:>15.4f} {abs(train_precision-test_precision):>15.4f}")
print(f"{'Recall':<15} {train_recall:>15.4f} {test_recall:>15.4f} {abs(train_recall-test_recall):>15.4f}")
print(f"{'F1 Score':<15} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1-test_f1):>15.4f}")
print("=" * 70)


print("=" * 70)
print(classification_report(
    y_test, y_test_pred,
    target_names=[f"Stage {c}" for c in le_target.classes_],
    zero_division=0
))



In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import time
import warnings

warnings.filterwarnings('ignore')

# CatBoost
cat_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.01,
    depth=5,
    loss_function='MultiClass',
    random_seed=42,
    verbose=20,
    thread_count=-1
)

print("CatBoost...")

start_time_cat = time.time()

cat_model.fit(
    X_train_bal, y_train_bal,
    eval_set=(X_test_scaled, y_test),
    verbose=20
)

cat_training_time = time.time() - start_time_cat

print(f"CatBoost training completed")
print(f"Total training time: {cat_training_time:.2f} seconds ({cat_training_time/60:.2f} minutes)")
print(f"Actual training iterations: {cat_model.tree_count_}")

train_score = cat_model.score(X_train_bal, y_train_bal)
test_score = cat_model.score(X_test_scaled, y_test)
print(f"Training set accuracy: {train_score:.6f}")
print(f"Test set accuracy: {test_score:.6f}")

best_iteration = cat_model.get_best_iteration()
print(f"Best iteration: {best_iteration}")


# Meta-learner


tabnet_train_proba = tabnet_model.predict_proba(X_train_bal)
cat_train_proba = cat_model.predict_proba(X_train_bal)

print(f"  TabNet output shape: {tabnet_train_proba.shape}")
print(f"  CatBoost output shape: {cat_train_proba.shape}")

meta_train_features = np.hstack([tabnet_train_proba, cat_train_proba])
print(f"  Meta-learner input shape: {meta_train_features.shape}")

meta_learner_cat = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs',
    verbose=1
)

start_time_meta = time.time()

meta_learner_cat.fit(meta_train_features, y_train_bal)

meta_training_time = time.time() - start_time_meta

print(f"Meta-learner training completed")
print(f"  Training time: {meta_training_time:.2f} seconds")
print(f"  Number of iterations: {meta_learner_cat.n_iter_[0]}")

#  Cross-validation


from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import cross_val_score

class StackedEnsembleCat(BaseEstimator, ClassifierMixin):

    def __init__(self, device='cpu', random_state=42):
        self.device = device
        self.random_state = random_state

    def fit(self, X, y):

        self.tabnet = TabNetClassifier(
            n_d=32, n_a=32, n_steps=3, gamma=1.3, lambda_sparse=1e-3,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=0.02),
            seed=self.random_state,
            device_name=self.device,
            verbose=0
        )
        self.tabnet.fit(
            X.values if hasattr(X, 'values') else X, y,
            max_epochs=100, patience=20, batch_size=256,
            virtual_batch_size=128
        )

        self.cat = CatBoostClassifier(
            iterations=200, learning_rate=0.01, depth=5,
            loss_function='MultiClass',
            random_seed=self.random_state, verbose=0,
            thread_count=-1
        )
        self.cat.fit(X, y)

        X_arr = X.values if hasattr(X, 'values') else X
        tabnet_proba = self.tabnet.predict_proba(X_arr)
        cat_proba = self.cat.predict_proba(X)
        meta_features = np.hstack([tabnet_proba, cat_proba])

        self.meta = LogisticRegression(
            C=1.0, max_iter=1000, random_state=self.random_state,
            multi_class='multinomial', solver='lbfgs', verbose=0
        )
        self.meta.fit(meta_features, y)

        return self

    def predict(self, X):
        X_arr = X.values if hasattr(X, 'values') else X
        tabnet_proba = self.tabnet.predict_proba(X_arr)
        cat_proba = self.cat.predict_proba(X)
        meta_features = np.hstack([tabnet_proba, cat_proba])
        return self.meta.predict(meta_features)


ensemble_cv_cat = StackedEnsembleCat(device=device, random_state=42)

cv_start_time = time.time()
cv_scores_cat = cross_val_score(
    ensemble_cv_cat, X_selected, y_encoded,
    cv=5, scoring='accuracy', n_jobs=1
)
cv_time_cat = time.time() - cv_start_time


for i, score in enumerate(cv_scores_cat, 1):
    print(f"  Fold {i:2d}: {score:.4f}")
print(f"\n{'='*40}")
print(f"  Average accuracy: {cv_scores_cat.mean():.4f}")
print(f"  Standard deviation: {cv_scores_cat.std():.4f}")
print(f"  Confidence interval: [{cv_scores_cat.mean()-1.96*cv_scores_cat.std():.4f}, {cv_scores_cat.mean()+1.96*cv_scores_cat.std():.4f}]")


# Feature importance analysis


cat_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': cat_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nCatBoost:")
print(cat_importance.to_string(index=False))


# Model evaluation


tabnet_train_test_proba = tabnet_model.predict_proba(X_train_scaled)
cat_train_test_proba = cat_model.predict_proba(X_train_scaled)
meta_train_test_features = np.hstack([tabnet_train_test_proba, cat_train_test_proba])
y_train_pred_cat = meta_learner_cat.predict(meta_train_test_features)

train_accuracy_cat = accuracy_score(y_train, y_train_pred_cat)
train_precision_cat = precision_score(y_train, y_train_pred_cat, average='weighted', zero_division=0)
train_recall_cat = recall_score(y_train, y_train_pred_cat, average='weighted', zero_division=0)
train_f1_cat = f1_score(y_train, y_train_pred_cat, average='weighted', zero_division=0)

tabnet_test_proba = tabnet_model.predict_proba(X_test_scaled)
cat_test_proba = cat_model.predict_proba(X_test_scaled)
meta_test_features = np.hstack([tabnet_test_proba, cat_test_proba])
y_test_pred_cat = meta_learner_cat.predict(meta_test_features)

test_accuracy_cat = accuracy_score(y_test, y_test_pred_cat)
test_precision_cat = precision_score(y_test, y_test_pred_cat, average='weighted', zero_division=0)
test_recall_cat = recall_score(y_test, y_test_pred_cat, average='weighted', zero_division=0)
test_f1_cat = f1_score(y_test, y_test_pred_cat, average='weighted', zero_division=0)


print("\n" + "=" * 70)
print(" " * 20 + "Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy_cat:>15.4f} {test_accuracy_cat:>15.4f} {abs(train_accuracy_cat-test_accuracy_cat):>15.4f}")
print(f"{'Precision':<15} {train_precision_cat:>15.4f} {test_precision_cat:>15.4f} {abs(train_precision_cat-test_precision_cat):>15.4f}")
print(f"{'Recall':<15} {train_recall_cat:>15.4f} {test_recall_cat:>15.4f} {abs(train_recall_cat-test_recall_cat):>15.4f}")
print(f"{'F1 Score':<15} {train_f1_cat:>15.4f} {test_f1_cat:>15.4f} {abs(train_f1_cat-test_f1_cat):>15.4f}")
print("=" * 70)


print("=" * 70)
print(classification_report(
    y_test, y_test_pred_cat,
    target_names=[str(c) for c in le_target.classes_],
    zero_division=0
))

